# Stage 2b: Window-Averaged Embeddings from Pre-computed CSV

Reads per-detection features from `batdetect2_detections.csv` (created by
`embed_stage1_batdetect.ipynb`) and builds a Hoplite embedding database with
one 32-dim average embedding per 0.5s window.

- **Windows with detections**: average raw features, then L2-normalise
- **Windows with NO detections**: sentinel vector `np.full(32, -1.0)`


## 1. Environment Setup

In [1]:
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

In [2]:
# Core imports
import os
import shutil
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from ml_collections import config_dict
from tqdm.auto import tqdm

# Hoplite DB
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.db import interface as db_interface
from perch_hoplite.agile import source_info

print("Imports successful! (No BatDetect2 needed — reading from CSV)")

Imports successful! (No BatDetect2 needed — reading from CSV)


## 2. Configuration

In [3]:
# USER CONFIGURATION

# Stage 1 CSV files (created by embed_stage1_batdetect.ipynb)
DETECTIONS_CSV = "C:/Users/Administrator/hello/Yucatan/batdetect2_detections.csv"
FILE_INDEX_CSV = "C:/Users/Administrator/hello/Yucatan/batdetect2_file_index.csv"

# Path for the Hoplite embedding database
DB_PATH = "C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings_BatDetect2"

# Audio source info (for hoplite metadata — must match Stage 1 config)
DATASET_NAME = "yucatan_bat_train"
DATASET_BASE_PATH = "C:/Users/Administrator/hello/Yucatan/acoustic_data/data/yucatan/audio"
FILE_GLOB = "*.wav"


# WINDOW CONFIGURATION



WINDOW_SIZE_S = 0.5
HOP_SIZE_S = 0.5 
EMBEDDING_DIM = 32
NORMALIZE_EMBEDDINGS = True

SENTINEL_VALUE = -1.0
SENTINEL_VECTOR = np.full(EMBEDDING_DIM, SENTINEL_VALUE, dtype=np.float32)

# DATABASE OPTIONS


DELETE_EXISTING_DB = True
USEARCH_DTYPE = 'float16'
USEARCH_METRIC = 'IP'


# STAGE 1 PARAMETERS 

TIME_EXPANSION = 1.0
DETECTION_THRESHOLD = 0.3
TARGET_SAMP_RATE = 256000
CHUNK_SIZE = 3.0

print("Configuration loaded.")
print(f"  Detections CSV: {DETECTIONS_CSV}")
print(f"  File index CSV: {FILE_INDEX_CSV}")
print(f"  Database path:  {DB_PATH}")
print(f"  Window size:    {WINDOW_SIZE_S}s (hop={HOP_SIZE_S}s)")
print(f"  Embedding dim:  {EMBEDDING_DIM}")
print(f"  Sentinel:       [{SENTINEL_VALUE}] * {EMBEDDING_DIM} (norm={np.linalg.norm(SENTINEL_VECTOR):.4f})")
print(f"  Normalise:      {NORMALIZE_EMBEDDINGS}")

Configuration loaded.
  Detections CSV: C:/Users/Administrator/hello/Yucatan/batdetect2_detections.csv
  File index CSV: C:/Users/Administrator/hello/Yucatan/batdetect2_file_index.csv
  Database path:  C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings_BatDetect2
  Window size:    0.5s (hop=0.5s)
  Embedding dim:  32
  Sentinel:       [-1.0] * 32 (norm=5.6569)
  Normalise:      True


## 3. Embedding Function

Groups detections by window, averages features, and emits sentinel for empty windows.
No BatDetect2 inference needed — just pandas grouping and numpy operations.

In [4]:
def embeddings_from_csv_windowed(
    file_df: pd.DataFrame,
    duration_s: float,
    feat_cols: List[str],
    window_size_s: float = 0.5,
    hop_size_s: float = 0.5,
    normalize: bool = True,
    sentinel: Optional[np.ndarray] = None,
) -> List[Tuple[float, np.ndarray, int]]:
    """Convert CSV rows for one file into window-averaged embeddings.

    For each window:
    - If detections exist: average raw features, L2-normalise
    - If no detections: emit sentinel vector

    Parameters:
        file_df: DataFrame rows for one source_id (from batdetect2_detections.csv)
        duration_s: File duration in seconds (from file_index)
        feat_cols: List of feature column names (feat_0 .. feat_31)
        window_size_s: Window size in seconds
        hop_size_s: Hop size in seconds
        normalize: If True, L2-normalise averaged embeddings
        sentinel: Vector to use for empty windows (None = skip empty windows)

    Returns:
        List of (window_offset_s, embedding, n_detections) tuples.
    """
    embedding_dim = len(feat_cols)
    results = []

   
    window_start = 0.0
    while window_start < duration_s:
        window_end = window_start + window_size_s

      
        remaining = duration_s - window_start
        if remaining < 0.1:
            break

      
        mask = (file_df["det_start_time"] >= window_start) & (file_df["det_start_time"] < window_end)
        window_dets = file_df[mask]
        n_det = len(window_dets)

        if n_det > 0:
            # Average features across detections in this window
            feats = window_dets[feat_cols].values.astype(np.float32)
            avg_feat = feats.mean(axis=0)

            if normalize:
                norm = np.linalg.norm(avg_feat)
                if norm > 0:
                    avg_feat = avg_feat / norm

            results.append((window_start, avg_feat, n_det))
        elif sentinel is not None:
            # Emit sentinel for empty window (NOT normalised)
            results.append((window_start, sentinel.copy(), 0))

        window_start += hop_size_s

    return results


print("embeddings_from_csv_windowed() defined")
print(f"  Empty windows: sentinel = [{SENTINEL_VALUE}] * {EMBEDDING_DIM}")

embeddings_from_csv_windowed() defined
  Empty windows: sentinel = [-1.0] * 32


## 4. Load and Inspect Stage 1 CSV

In [5]:
# Load detections CSV
df = pd.read_csv(DETECTIONS_CSV)
print(f"Loaded {len(df)} detections from {DETECTIONS_CSV}")
print(f"Unique files: {df['source_id'].nunique()}")

# Load file index (needed for file durations)
file_index = pd.read_csv(FILE_INDEX_CSV)
print(f"Loaded {len(file_index)} files from {FILE_INDEX_CSV}")

# Build duration lookup
duration_lookup = dict(zip(file_index["source_id"], file_index["duration_s"]))
print(f"Duration lookup: {len(duration_lookup)} files")
print(f"Duration range: [{file_index['duration_s'].min():.3f}, {file_index['duration_s'].max():.3f}]s")

# Feature columns
FEAT_COLS = [f"feat_{i}" for i in range(EMBEDDING_DIM)]

# Test on one file
test_file = file_index["source_id"].iloc[0]
test_dur = duration_lookup[test_file]
test_df = df[df["source_id"] == test_file]
test_embs = embeddings_from_csv_windowed(
    test_df, test_dur, FEAT_COLS,
    window_size_s=WINDOW_SIZE_S, hop_size_s=HOP_SIZE_S,
    normalize=NORMALIZE_EMBEDDINGS, sentinel=SENTINEL_VECTOR,
)
n_with_det = sum(1 for _, _, n in test_embs if n > 0)
n_sentinel = sum(1 for _, _, n in test_embs if n == 0)
print(f"\nTest file: {test_file}")
print(f"  Duration: {test_dur:.3f}s")
print(f"  Detections in CSV: {len(test_df)}")
print(f"  Windows: {len(test_embs)} ({n_with_det} with detections, {n_sentinel} sentinel)")
if test_embs:
    off, emb, nd = test_embs[0]
    tag = "detection-averaged" if nd > 0 else "SENTINEL"
    print(f"  First window: offset={off:.3f}s, n_det={nd}, norm={np.linalg.norm(emb):.4f} ({tag})")

Loaded 13188 detections from C:/Users/Administrator/hello/Yucatan/batdetect2_detections.csv
Unique files: 1159
Loaded 1395 files from C:/Users/Administrator/hello/Yucatan/batdetect2_file_index.csv
Duration lookup: 1395 files
Duration range: [0.700, 1.000]s

Test file: dry_azul_1_110304_2000_3000.wav
  Duration: 1.000s
  Detections in CSV: 7
  Windows: 2 (2 with detections, 0 sentinel)
  First window: offset=0.000s, n_det=5, norm=1.0000 (detection-averaged)


## 5. Initialize Hoplite Database

In [6]:
# Handle existing database
db_path = Path(DB_PATH)

if db_path.exists():
    if DELETE_EXISTING_DB:
        print(f"Deleting existing database at {DB_PATH}...")
        shutil.rmtree(db_path)
    else:
        raise ValueError(
            f"Database exists at {DB_PATH}. "
            "Set DELETE_EXISTING_DB=True to overwrite."
        )

# USearch configuration
usearch_cfg = config_dict.ConfigDict({
    'embedding_dim': EMBEDDING_DIM,
    'dtype': USEARCH_DTYPE,
    'metric_name': USEARCH_METRIC,
    'expansion_add': 256,
    'expansion_search': 128,
})

print(f"Creating database with:")
print(f"  - Embedding dim: {usearch_cfg.embedding_dim}")
print(f"  - Dtype: {usearch_cfg.dtype}")
print(f"  - Metric: {usearch_cfg.metric_name}")

# Create database
db = sqlite_usearch_impl.SQLiteUsearchDB.create(
    db_path=str(db_path),
    usearch_cfg=usearch_cfg
)

print(f"\n\u2713 Database created at {DB_PATH}")

Deleting existing database at C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings_BatDetect2...
Creating database with:
  - Embedding dim: 32
  - Dtype: float16
  - Metric: IP

✓ Database created at C:/Users/Administrator/hello/Yucatan/Yucatan_embeddings_BatDetect2


## 6. Store Metadata

In [7]:
# Store model configuration metadata
model_config_dict = config_dict.ConfigDict({
    'model_key': 'batdetect2',
    'model_config': {
        'sample_rate': TARGET_SAMP_RATE,
        'embedding_dim': EMBEDDING_DIM,
        'feature_type': 'cnn_32dim_window_avg',
        'extraction_method': 'stage1_csv',
        'stage1_csv': DETECTIONS_CSV,
        'window_size_s': WINDOW_SIZE_S,
        'hop_size_s': HOP_SIZE_S,
        'detection_threshold': DETECTION_THRESHOLD,
        'time_expansion': TIME_EXPANSION,
        'target_samp_rate': TARGET_SAMP_RATE,
        'chunk_size': CHUNK_SIZE,
        'normalize_embeddings': NORMALIZE_EMBEDDINGS,
        'sentinel_value': float(SENTINEL_VALUE),
    }
})
db.insert_metadata('model_config', model_config_dict)
print("Stored model_config metadata")

# Store audio source configuration
audio_source = source_info.AudioSourceConfig(
    dataset_name=DATASET_NAME,
    base_path=DATASET_BASE_PATH,
    file_glob=FILE_GLOB,
)
audio_sources = source_info.AudioSources((audio_source,))
audio_sources_dict = config_dict.ConfigDict(audio_sources.to_config_dict())
db.insert_metadata('audio_sources', audio_sources_dict)
print("Stored audio_sources metadata")

print("\n--- Metadata ---")
for k, v in model_config_dict.model_config.items():
    print(f"  {k}: {v}")

Stored model_config metadata
Stored audio_sources metadata

--- Metadata ---
  chunk_size: 3.0
  detection_threshold: 0.3
  embedding_dim: 32
  extraction_method: stage1_csv
  feature_type: cnn_32dim_window_avg
  hop_size_s: 0.5
  normalize_embeddings: True
  sample_rate: 256000
  sentinel_value: -1.0
  stage1_csv: C:/Users/Administrator/hello/Yucatan/batdetect2_detections.csv
  target_samp_rate: 256000
  time_expansion: 1.0
  window_size_s: 0.5


## 7. Batch Embedding Extraction

In [8]:

# Batch insertion to hoplite

stats = {
    'files_processed': 0,
    'files_with_detections': 0,
    'total_windows': 0,
    'windows_with_detections': 0,
    'windows_sentinel': 0,
    'total_detections_used': 0,
    'errors': [],
}

# Group detections by source_id for efficient lookup
det_grouped = df.groupby("source_id", sort=False)

# Process ALL files from file_index (including those with 0 detections)
print(f"Processing {len(file_index)} files...")
print(f"Window: {WINDOW_SIZE_S}s, hop: {HOP_SIZE_S}s")
print(f"Sentinel for empty windows: [{SENTINEL_VALUE}] * {EMBEDDING_DIM}")

for _, row in tqdm(file_index.iterrows(), desc="Stage 2b (CSV→DB)", unit="file", total=len(file_index)):
    source_id = row["source_id"]
    duration_s = row["duration_s"]

    try:
        # Get detections for this file (empty DataFrame if none)
        if source_id in det_grouped.groups:
            file_df = det_grouped.get_group(source_id)
        else:
            file_df = pd.DataFrame(columns=df.columns)

        # Generate window embeddings
        windows = embeddings_from_csv_windowed(
            file_df, duration_s, FEAT_COLS,
            window_size_s=WINDOW_SIZE_S, hop_size_s=HOP_SIZE_S,
            normalize=NORMALIZE_EMBEDDINGS, sentinel=SENTINEL_VECTOR,
        )

        # Insert each window embedding
        for offset_s, embedding, n_det in windows:
            offsets = np.asarray([offset_s], dtype=np.float16)

            source = db_interface.EmbeddingSource(
                dataset_name=DATASET_NAME,
                source_id=source_id,
                offsets=offsets,
            )

            db.insert_embedding(
                np.asarray(embedding, dtype=np.float32), source
            )

            stats['total_windows'] += 1
            if n_det > 0:
                stats['windows_with_detections'] += 1
                stats['total_detections_used'] += n_det
            else:
                stats['windows_sentinel'] += 1

        stats['files_processed'] += 1
        if len(file_df) > 0:
            stats['files_with_detections'] += 1

    except Exception as e:
        stats['errors'].append((source_id, str(e)))
        if len(stats['errors']) <= 5:
            tqdm.write(f"[ERROR] {source_id}: {e}")

# Commit
db.commit()

print(f"\n{'='*50}")
print("EMBEDDING COMPLETE")
print(f"{'='*50}")

Processing 1395 files...
Window: 0.5s, hop: 0.5s
Sentinel for empty windows: [-1.0] * 32


Stage 2b (CSV→DB):   0%|          | 0/1395 [00:00<?, ?file/s]


EMBEDDING COMPLETE
